In [5]:
!pip install -U selenium
!pip install webdriver_manager
!pip install fake-useragent
!pip install undetected-chromedriver


from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
import time

# 建立 Chrome 瀏覽器物件
driver = webdriver.Chrome()
driver.maximize_window()


     ---------------------------------------- 0.0/65.4 kB ? eta -:--:--
     ------ --------------------------------- 10.2/65.4 kB ? eta -:--:--
     ----------------------- -------------- 41.0/65.4 kB 393.8 kB/s eta 0:00:01
     -------------------------------------- 65.4/65.4 kB 506.1 kB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/176.8 kB ? eta -:--:--
   -------------------- ------------------- 92.2/176.8 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 176.8/176.8 kB 2.6 MB/s eta 0:00:00
  Created wheel for undetected-chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47130 sha256=6c0614c41c19f05a026d2c8e97e5c46c3fb5d1ace1a37dde9f84502b9da2e8e1
  Stored in directory: c:\users\a7890\appdata\local\pip\cache\wheels\c4\f1\aa\9de6cf276210554d91e9c0526864563e850a428c5e76da4914
Successfully built undetected-chromedriver


In [10]:
import time
import pickle
import undetected_chromedriver as uc

# 建立瀏覽器，開啟 Shopee 首頁
driver = uc.Chrome()
driver.get("https://shopee.tw/")

print("請在此瀏覽器視窗中手動登入 Shopee（例如掃碼登入）。")
input("登入完成後請按 Enter...")

# 取得登入後的 Cookies
cookies = driver.get_cookies()
with open("shopee_cookies.pkl", "wb") as f:
    pickle.dump(cookies, f)
print("✅ Cookies 已成功儲存到 shopee_cookies.pkl")

driver.quit()


請在此瀏覽器視窗中手動登入 Shopee（例如掃碼登入）。
✅ Cookies 已成功儲存到 shopee_cookies.pkl


In [14]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pickle
import time
import pandas as pd

# === 1. 設定瀏覽器 ===
options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

# === 2. 先進入首頁，載入 cookie ===
driver.get("https://shopee.tw/")
time.sleep(2)

with open("shopee_cookies.pkl", "rb") as f:
    cookies = pickle.load(f)
    for cookie in cookies:
        if "expiry" in cookie:
            del cookie["expiry"]
        driver.add_cookie(cookie)

# === 3. 重新導向搜尋頁面 ===
driver.get("https://shopee.tw/search?keyword=滑鼠")
time.sleep(5)

# === 4. 緩慢下滑，讓商品載入 ===
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollBy(0, 1000);")
    time.sleep(1.5)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

# === 5. 等商品名稱區塊出現 ===
WebDriverWait(driver, 15).until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.line-clamp-2.text-sm"))
)

# === 6. 抓取商品名稱（略過圖片）===
elements = driver.find_elements(By.CSS_SELECTOR, "div.line-clamp-2.text-sm")

results = []
for el in elements:
    # 只取純文字部分，不包括圖片
    text_nodes = driver.execute_script("""
        let el = arguments[0];
        let texts = [];
        for (let node of el.childNodes) {
            if (node.nodeType === Node.TEXT_NODE) {
                texts.push(node.textContent.trim());
            }
        }
        return texts.join(' ');
    """, el)

    if text_nodes:
        results.append({"商品名稱": text_nodes})

# === 7. 存成 CSV ===
df = pd.DataFrame(results)
df.to_csv("shopee_滑鼠商品名稱.csv", index=False, encoding="utf-8-sig")
print("✅ 共儲存", len(df), "筆商品名稱")
driver.quit()


✅ 共儲存 63 筆商品名稱
